# Chatbot with Long Term Store Memory , TrustCall and Collection Schema

## Review:
- previously we explored short term and Long term memory, Memory store and trustcall
### short-term Memory : 
- Memory limited to a session or thread.
- Memory is lost when the session ends
### long-term Memory :
#### Multi session DB
- Simple multi session memory stored in a DB (SQLite, PostGress)
### multi session/thread store
-  Memory is stored in a store as
      - single user profile JSON key and value pair 
      - memory is updated either by regenerating while updating or as a JSON patch to existing memory
- Uses put, get and search method to manipulate the store
- important objects are namespace, key and value
- uses TrustCall to avoid LLm regenerating memory with every update to reduce cost and latency
- trust call only generates JSON objects for new and updated information while preserving the exiting memory.

## Goal:
-  Save memory as collection/List of profiles or documents
    - Allows for smaller, narrowly scoped memories w/ easy addition of new information
- use TrustCall to generate JSON patch of new information to update the existing memory

In [1]:
# Load API key
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_USE_V1"] = "true"


In [2]:
# create genai client and llm
from google import genai

client = genai.Client(api_key = os.environ["GOOGLE_API_KEY"])
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [3]:
# create a llm using any of the above models
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI( model= "gemini-flash-lite-latest" , 
                              temperature = 0.2 )
llm.invoke("What day is this?").content

[{'type': 'text',
  'text': 'Today is **Wednesday, May 22, 2024**.',
  'extras': {'signature': 'EjQKMgEMOdbHyP+tMG1Mlx0032sz7HDyiOMT9Lqzcz3bqJSBwFecM/FEgfetsjOmBE4LROrW'}}]

## Defining Collection Schema

- Instead of updating the profiel object every time, collection save the new information as new JSON doc .
-EX:
1. What a “collection schema” is
Instead of:
```
# One big profile
{
  "user_name": "Diya",
  "age": 31,
  "interests": ["gardening", "hiking"],
  ...
}
```
You store many small entries, each one a single memory:
```
# Many small docs
{
  "content": "User likes gardening",
  "memory_type": "preference",
  "source": "chat",
  "importance": 0.7,
}
{
  "content": "User lives in San Antonio",
  "memory_type": "fact",
  "source": "chat",
  "importance": 0.9,
}
```
- Each of these is one row in your “memories collection”.
- Each memory will be stored as a separate entry with a single content field for the main information we want to remember
- Use collection schema as a struvtured output

Pros:
- Avoids updating the whole profile
- model may overwrite or change information
- tracking and observability isnt possible
- can not store ambiguous , open ended facts
- avoids cost and latency



In [56]:
# Example:

# define the collection schema as pydantic BaseModel
from pydantic import BaseModel ,  Field
from typing import List
from langchain_core.messages import HumanMessage , AIMessage , SystemMessage

# define memory
class Memory(BaseModel):
    content: str = Field(description="Content of the memory .For Example: User said he likes to travel to Europe")

#Define main memory collection schema
class Memory_collection(BaseModel):
    memories: List[Memory] = Field(description="list of memories about the user")

# set up chat messages
msg = HumanMessage(content="Hi, Im Diya. I love reading books. I would love to visit the Library tomorrow")

# get response from the LLM
response = llm.with_structured_output(Memory_collection).invoke([msg])
response.memories

[Memory(content="User's name is Diya."),
 Memory(content='Diya loves reading books.'),
 Memory(content='Diya plans to visit the library tomorrow.')]

## model_dump() - serialize pydantic model instance into a dictionary

In [57]:
for m in response.memories:
    print(m.model_dump())

{'content': "User's name is Diya."}
{'content': 'Diya loves reading books.'}
{'content': 'Diya plans to visit the library tomorrow.'}


### Store Memory
- store memory as a collection of dictionary objects with different keys

In [58]:
# define user_id and  namespace 
user_id = "1"
namespace = ('memories' , user_id)
# get key
import uuid


# set store
from langgraph.store.memory import InMemoryStore
store_memory = InMemoryStore()

# store memory objects as individual values with different keys
for m in response.memories:
    value = m.model_dump()
    print(value)
    key  = str(uuid.uuid4())
    print(key)
    store_memory.put(namespace, key , value)

    

{'content': "User's name is Diya."}
60d50ca5-a280-4b15-9332-d1c07660a0d1
{'content': 'Diya loves reading books.'}
52bc4fc3-2756-4039-9b54-9d942b353901
{'content': 'Diya plans to visit the library tomorrow.'}
1a1b455d-a0de-4ef4-8c87-5a0f676a0325


## check for memory
- Notice that memory is a collection of objects
- each collection is a individual key and value pair
- 


In [59]:
memory = store_memory.search(namespace)
for m in memory:
    print(m.dict())
    print("\n")

{'namespace': ['memories', '1'], 'key': '60d50ca5-a280-4b15-9332-d1c07660a0d1', 'value': {'content': "User's name is Diya."}, 'created_at': '2026-06-07T20:52:15.886623+00:00', 'updated_at': '2026-06-07T20:52:15.886623+00:00', 'score': None}


{'namespace': ['memories', '1'], 'key': '52bc4fc3-2756-4039-9b54-9d942b353901', 'value': {'content': 'Diya loves reading books.'}, 'created_at': '2026-06-07T20:52:15.888630+00:00', 'updated_at': '2026-06-07T20:52:15.888630+00:00', 'score': None}


{'namespace': ['memories', '1'], 'key': '1a1b455d-a0de-4ef4-8c87-5a0f676a0325', 'value': {'content': 'Diya plans to visit the library tomorrow.'}, 'created_at': '2026-06-07T20:52:15.888630+00:00', 'updated_at': '2026-06-07T20:52:15.888630+00:00', 'score': None}




## Upading collection schema
- update the collection with new memory as well existing collection
- use TrustaCall to avoid overwriting the collection everytime (cost , latency , token usage).
- TrustCall generates JSON patch and creats new collection and updates existing collection

### TrustCall memory extractor
- create a trustcall extractor
- 1. Extract memory as a single object
- 2. Extract memory as a collection

### 1. Single profile |memory

In [73]:
# create a truscall extractor
from trustcall import create_extractor

trustcall_extractor = create_extractor(
    llm,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True
)


# set up chat messages
from langchain_core.messages import HumanMessage, AIMessage , SystemMessage



conversation = [HumanMessage(content="Hi, THis is Diya. I lvoe gardening. I want to find a nursary to buy plants"), 
                AIMessage(content="Nice to meet you Diya. That's great!. what kind of plants do you like?"),
                HumanMessage(content="I love flower plants. I like colors"),
               ]

system_msg = SystemMessage(content="you are a memory extractor for collection and storing.  extract memories from the chat messages given below")

#extract memory
result = trustcall_extractor.invoke({'messages' : [system_msg] + conversation })


In [74]:
result

{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'Memory', 'arguments': '{"content": "User\'s name is Diya. She loves gardening, specifically flower plants, and enjoys colorful plants. She is looking for a nursery to buy plants."}'}, '__gemini_function_call_thought_signatures__': {'7cb7b295-627c-4d9e-98bc-486a50b9b3ae': 'EjQKMgEMOdbHNwWvAoxgqVaUxhfkq9X/Q/0KQrPzAYJ1mouOUCcECgNM4rmktzhJJNnoxF0c'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ea3e1-9cd7-7101-ba45-670b93a76631-0', tool_calls=[{'name': 'Memory', 'args': {'content': "User's name is Diya. She loves gardening, specifically flower plants, and enjoys colorful plants. She is looking for a nursery to buy plants."}, 'id': '7cb7b295-627c-4d9e-98bc-486a50b9b3ae', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 44, 'total_tokens': 175, 'input_

In [75]:
result['responses'][-1].model_dump()

{'content': "User's name is Diya. She loves gardening, specifically flower plants, and enjoys colorful plants. She is looking for a nursery to buy plants."}

In [76]:
id = result['messages'][-1].tool_calls[-1]['id']
id

'7cb7b295-627c-4d9e-98bc-486a50b9b3ae'

### 2. Collection of TrustCall memory objects 

In [77]:
trustcall_extractor = create_extractor(
    llm,
    tools=[Memory_collection],
    tool_choice="Memory_collection",
    enable_inserts=True
)


# set up chat messages
from langchain_core.messages import HumanMessage, AIMessage , SystemMessage



conversation = [HumanMessage(content="Hi, THis is Diya. I love gardening. I want to find a nursary to buy plants"), 
                AIMessage(content="Nice to meet you Diya. That's great!. what kind of plants do you like?"),
                HumanMessage(content="I love flower plants. I like colors"),
               ]

system_msg = SystemMessage(content="you are a memory extractor for collection and storing.  extract memories from the chat messages given below")

#extract memory
result = trustcall_extractor.invoke({'messages' : [system_msg] + conversation })

Key '$defs' is not supported in schema, ignoring


## ⭐ Why TrustCall cannot extract lists
TrustCall internally:
- Validates a single schema
- Generates a JSON Patch for a single object
- Applies updates to a single object

#### It cannot:
- Patch a list
- Insert into a list
- Validate nested models
- Handle $defs from Pydantic

#### So this will never work:
```
class Memory_collection(BaseModel):
    memories: List[memory]
```
TrustCall will always ignore $defs and fail.

## ⭐ The correct LangGraph pattern (recommended by LangChain team)
✔ Extract one memory
✔ Store it as one document
✔ Build a collection over time

### Example:
```
result = trustcall_extractor.invoke({...})

if result.content.strip():
    store.put(namespace, None, result.model_dump())
```
Passing `None` lets the store auto‑generate a unique key.

## This is exactly how:
- MemoryGPT
- LangGraph course examples
- ReAct agents
- Long‑term memory agents
- are built.

In [78]:
result

{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'Memory_collection', 'arguments': '{"memories": [{"content": "User\'s name is Diya."}, {"content": "Diya loves gardening."}, {"content": "Diya loves flower plants and likes colors."}]}'}, '__gemini_function_call_thought_signatures__': {'2a52976b-c670-4d91-92d8-9134c7ebe38f': 'EjQKMgEMOdbHoO3d/y126V9G8GeYKM/swijJPX0tNqfig1ZemUGij0bRJfXp8mx8ZGOi3u1t'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ea3e2-a21c-72c2-97cd-1c13fc8f8751-0', tool_calls=[{'name': 'Memory_collection', 'args': {'memories': [{'content': "User's name is Diya."}, {'content': 'Diya loves gardening.'}, {'content': 'Diya loves flower plants and likes colors.'}]}, 'id': '2a52976b-c670-4d91-92d8-9134c7ebe38f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 176, 'output_tokens': 51, 'total_tokens': 227, '